In [1]:
import os
import re
import sys
import sys 
import numpy as np
import pandas as pd
import torch
from scipy.signal import find_peaks
from scipy import stats
from statsmodels.stats.multitest import multipletests
import matplotlib
matplotlib.use("Agg")   # non-interactive backend, avoids the inline backend bug
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline,Pipeline
from sklearn.metrics import f1_score,mean_absolute_error, mean_squared_error,r2_score
from sklearn.model_selection import KFold, cross_val_score, GroupKFold
from sklearn.compose import ColumnTransformer

from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
# !pip install catboost
from catboost import CatBoostRegressor
import warnings
warnings.filterwarnings("ignore")
from sklearn.linear_model import RidgeCV
from sklearn.decomposition import PCA

In [2]:
sys.path.append("/data1/yashvi_bhuva/anemia/PPG_BP/PaPaGei/papagei-foundation-model")

import pyPPG.preproc as PP
from dotmap import DotMap
from linearprobing.utils import resample_batch_signal, load_model_without_module_prefix
from models.resnet import ResNet1DMoE

In [3]:
RAW_DIR = "/data1/yashvi_bhuva/anemia/PPG_BP/ppg_data"             # folder of raw .txt files, e.g. "2_1.txt"
FS_ORIGINAL = 1000                 # PPG-BP native sampling rate — confirm
FS_TARGET_PAPAGEI = 125            # PaPaGei-S expected input rate
WEIGHTS_PATH = "papagei-foundation-model/weights/papagei_s.pt"
METADATA_PATH = "/data1/yashvi_bhuva/anemia/PPG_BP/PPG-BP dataset.xlsx"

fname_pattern = re.compile(r"^(\d+)_(\d+)")


meta = pd.read_excel(METADATA_PATH, sheet_name="cardiovascular dataset", header=1)
meta["subject_ID"] = meta["subject_ID"].astype(int)
meta = meta.set_index("subject_ID")

# Features to be extracted from PPG

def get_peaks(ppg, fs):
    peaks, _ = find_peaks(ppg, distance=int(0.4 * fs))
    return peaks

def heart_rate_feature(ppg, fs):
    peaks = get_peaks(ppg, fs)
    if len(peaks) < 2:
        return np.nan
    ppi = np.diff(peaks) / fs
    return 60 / np.mean(ppi)

def mean_peak_amplitude_feature(ppg, fs):
    peaks = get_peaks(ppg, fs)
    return np.mean(ppg[peaks]) if len(peaks) > 0 else np.nan

def std_peak_amplitude_feature(ppg, fs):
    peaks = get_peaks(ppg, fs)
    return np.std(ppg[peaks]) if len(peaks) > 0 else np.nan

def mean_peak_distance_feature(ppg, fs):
    peaks = get_peaks(ppg, fs)
    return np.mean(np.diff(peaks)) if len(peaks) >= 2 else np.nan

def extract_features(ppg, vpg, apg, fs):
    features = {
        "mean_ppg": np.mean(ppg), "std_ppg": np.std(ppg),
        "skew_ppg": stats.skew(ppg), "kurtosis_ppg": stats.kurtosis(ppg),
        "rms_ppg": np.sqrt(np.mean(ppg**2)), "max_ppg": np.max(ppg),
        "min_ppg": np.min(ppg), "range_ppg": np.ptp(ppg),
        "heart_rate": heart_rate_feature(ppg, fs),
        "mean_peak_amp": mean_peak_amplitude_feature(ppg, fs),
        "std_peak_amp": std_peak_amplitude_feature(ppg, fs),
        "mean_peak_distance": mean_peak_distance_feature(ppg, fs),
        "max_vpg": np.max(vpg), "min_vpg": np.min(vpg),
        "mean_vpg": np.mean(vpg), "std_vpg": np.std(vpg),
        "max_apg": np.max(apg), "min_apg": np.min(apg),
        "mean_apg": np.mean(apg), "std_apg": np.std(apg),
        "dominant_freq": np.nan, "spectral_energy": np.sum(ppg**2),
    }
    freqs = np.fft.rfftfreq(len(ppg), d=1/fs)
    fft_mag = np.abs(np.fft.rfft(ppg))
    if len(fft_mag) > 1:
        features["dominant_freq"] = freqs[1:][np.argmax(fft_mag[1:])]
    return features


# The below function cleans the raw PPG signal to match what PaPaGei was trained on
def preprocess_one_ppg_signal(waveform, frequency, fL=0.5, fH=12, order=4,
                                smoothing_windows={"ppg": 50, "vpg": 10, "apg": 10, "jpg": 10}):
    prep = PP.Preprocess(fL=fL, fH=fH, order=order, sm_wins=smoothing_windows)
    signal = DotMap()
    signal.v = waveform
    signal.fs = frequency
    signal.filtering = True
    ppg, ppg_d1, ppg_d2, ppg_d3 = prep.get_signals(signal)
    return ppg, ppg_d1, ppg_d2, ppg_d3


def is_signal_flat_lined_simple(sig, fs, flat_threshold=0.25, change_threshold=0.01, window_ms=20):
    sig = np.asarray(sig, dtype=float)
    sig_norm = (sig - np.mean(sig)) / (np.std(sig) + 1e-8)

    step = max(1, int(fs * window_ms / 1000))  # compare samples ~20ms apart, not adjacent
    diffs = np.abs(sig_norm[step:] - sig_norm[:-step])
    flat_fraction = np.sum(diffs < change_threshold) / len(diffs)
    return 1 if flat_fraction > flat_threshold else 0

In [4]:
# Creating the PaPaGei model and loading weights
model_config = {
    'base_filters': 32, 'kernel_size': 3, 'stride': 2, 'groups': 1,
    'n_block': 18, 'n_classes': 512, 'n_experts': 3
}
papagei_model = ResNet1DMoE(in_channels=1, **model_config)
papagei_model = load_model_without_module_prefix(papagei_model, WEIGHTS_PATH)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
papagei_model.to(device).eval()

cuda


ResNet1DMoE(
  (first_block_conv): MyConv1dPadSame(
    (conv): Conv1d(1, 32, kernel_size=(3,), stride=(1,))
  )
  (first_block_bn): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (first_block_relu): ReLU()
  (basicblock_list): ModuleList(
    (0): BasicBlock(
      (bn1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu1): ReLU()
      (do1): Dropout(p=0.5, inplace=False)
      (conv1): MyConv1dPadSame(
        (conv): Conv1d(32, 32, kernel_size=(3,), stride=(1,))
      )
      (bn2): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu2): ReLU()
      (do2): Dropout(p=0.5, inplace=False)
      (conv2): MyConv1dPadSame(
        (conv): Conv1d(32, 32, kernel_size=(3,), stride=(1,))
      )
      (max_pool): MyMaxPool1dPadSame(
        (max_pool): MaxPool1d(kernel_size=1, stride=1, padding=0, dilation=1, ceil_mode=False)
      )
    )
    

In [5]:
# Preprocess all raw PPG files, compute VPG& APG, extract features, pair with BP labels and save to CSV
log_rows = []
feature_rows = []
n_ok, n_fail, n_flatlined = 0, 0, 0

for file in os.listdir(RAW_DIR):
    try:
        match = fname_pattern.match(file)
        if not match:
            raise ValueError(f"Could not parse subject_id/segment from '{file}'")
        subject_id, segment = int(match.group(1)), int(match.group(2))
        if subject_id not in meta.index:
            raise KeyError(f"subject_id {subject_id} not in metadata")

        raw_ppg = np.loadtxt(os.path.join(RAW_DIR, file))

        # -- filter + derivatives (PaPaGei's exact preprocessing) --
        ppg_filt, vpg_filt, apg_filt, _ = preprocess_one_ppg_signal(
            waveform=raw_ppg, frequency=FS_ORIGINAL
        )

        # -- quality gate --
        if is_signal_flat_lined_simple(ppg_filt, fs=FS_ORIGINAL):
            n_flatlined += 1
            log_rows.append({"file": file, "status": "flatlined", "error": ""})
            continue

        # -- hand-crafted features (on the same filtered signal) --
        feature_row = extract_features(ppg_filt, vpg_filt, apg_filt, FS_ORIGINAL)

        # -- PaPaGei embedding --
        resampled = resample_batch_signal(
            ppg_filt[np.newaxis, :], fs_original=FS_ORIGINAL,
            fs_target=FS_TARGET_PAPAGEI, axis=-1
        )
        signal_tensor = torch.Tensor(resampled).unsqueeze(dim=1).to(device)
        with torch.inference_mode():
            outputs = papagei_model(signal_tensor)
            embedding = outputs[0].cpu().numpy().squeeze()
        for i, v in enumerate(embedding):
            feature_row[f"papagei_{i}"] = v

        # -- labels / demographics --
        label_row = meta.loc[subject_id]
        feature_row["SBP"] = label_row["Systolic Blood Pressure(mmHg)"]
        feature_row["DBP"] = label_row["Diastolic Blood Pressure(mmHg)"]
        feature_row["Age"] = label_row["Age(year)"]
        feature_row["BMI"] = label_row["BMI(kg/m^2)"]
        feature_row["Hypertension"] = label_row["Hypertension"]

        feature_row["file"] = file
        feature_row["subject_id"] = subject_id
        feature_row["segment"] = segment

        feature_rows.append(feature_row)
        log_rows.append({"file": file, "status": "ok", "error": ""})
        n_ok += 1

    except Exception as e:
        log_rows.append({"file": file, "status": "failed", "error": str(e)})
        n_fail += 1
        print(f"[FAILED] {file}: {e}")

print(f"\nDone. {n_ok} ok, {n_flatlined} flatlined (excluded), {n_fail} failed.")
features_df = pd.DataFrame(feature_rows)
log_df = pd.DataFrame(log_rows)
print(features_df.shape)



Done. 657 ok, 0 flatlined (excluded), 0 failed.
(657, 542)


In [6]:
hand_crafted_cols = ["mean_ppg","std_ppg","skew_ppg","kurtosis_ppg","rms_ppg","max_ppg",
    "min_ppg","range_ppg","heart_rate","mean_peak_amp","std_peak_amp","mean_peak_distance",
    "max_vpg","min_vpg","mean_vpg","std_vpg","max_apg","min_apg","mean_apg","std_apg",
    "dominant_freq","spectral_energy"]
papagei_cols = [c for c in features_df.columns if c.startswith("papagei_")]

In [7]:
# Checking how features are related to DBP using Ridge regression with GroupKFold cross-validation
data = features_df.dropna(subset=["DBP"]).copy()

# Target
y = data["DBP"].values

# Subject IDs for leakage-free splitting
groups = data["subject_id"].values
gkf = GroupKFold(n_splits=5)  #Subject-wise splitting to avoid data leakage

conditions = {
    "Demographics only":
        ["BMI", "Age"],
    "Hand-crafted PPG feats":
        hand_crafted_cols + ["BMI", "Age"],
    "PaPaGei embeddings":
        papagei_cols + ["BMI", "Age"],
    "All features":
        hand_crafted_cols + papagei_cols + ["BMI", "Age"]
        }

for name, cols in conditions.items():
    print("\n", name)
    X = data[cols].values
    model = make_pipeline(
        StandardScaler(),
        Ridge(alpha=1.0)
    )
    r2_scores = cross_val_score(
        model,
        X,
        y,
        cv=gkf,
        groups=groups,
        scoring="r2"
    )
    print(
        f"R² = {r2_scores.mean():.3f} ± {r2_scores.std():.3f}"
    )


 Demographics only
R² = -0.012 ± 0.090

 Hand-crafted PPG feats
R² = 0.012 ± 0.100

 PaPaGei embeddings
R² = -0.486 ± 0.182

 All features
R² = -0.523 ± 0.203


In [8]:
# Checking how features are related to SBP using Ridge regression with GroupKFold cross-validation
data = features_df.dropna(subset=["SBP"]).copy()

# Target
y = data["SBP"].values

# Subject IDs for leakage-free splitting
groups = data["subject_id"].values
gkf = GroupKFold(n_splits=5)  #Subject-wise splitting to avoid data leakage

conditions = {
    "Demographics only":
        ["BMI", "Age"],
    "Hand-crafted PPG feats":
        hand_crafted_cols + ["BMI", "Age"],
    "PaPaGei embeddings":
        papagei_cols + ["BMI", "Age"],
    "All features":
        hand_crafted_cols + papagei_cols + ["BMI", "Age"]
        }

for name, cols in conditions.items():
    print("\n", name)
    X = data[cols].values
    model = make_pipeline(
        StandardScaler(),
        Ridge(alpha=1.0)
    )
    r2_scores = cross_val_score(
        model,
        X,
        y,
        cv=gkf,
        groups=groups,
        scoring="r2"
    )
    print(
        f"R² = {r2_scores.mean():.3f} ± {r2_scores.std():.3f}"
    )


 Demographics only
R² = 0.138 ± 0.151

 Hand-crafted PPG feats
R² = 0.141 ± 0.155

 PaPaGei embeddings
R² = -0.459 ± 0.392

 All features
R² = -0.589 ± 0.571


In [9]:
# Checking how features are related to hypertension status using Ridge regression with GroupKFold cross-validation
hypertension_map = {
    "Normal": 0,
    "Prehypertension": 1,
    "Stage 1 hypertension": 2,
    "Stage 2 hypertension": 3
}

features_df["Hypertension_class"] = (
    features_df["Hypertension"]
    .map(hypertension_map)
)
# print(features_df["Hypertension_class"].value_counts())

def evaluate_hypertension():
    data = features_df.dropna(
        subset=["Hypertension_class"]
    ).copy()

    y = data["Hypertension_class"].values
    groups = data["subject_id"].values
    conditions = {
        "Demographics only":
            ["BMI", "Age"],
        "Hand-crafted PPG feats":
            hand_crafted_cols + ["BMI", "Age"],
        "PaPaGei embeddings":
            papagei_cols + ["BMI", "Age"],
        "All features":
            hand_crafted_cols + papagei_cols + ["BMI", "Age"]
    }
    gkf = GroupKFold(n_splits=5)
    for name, cols in conditions.items():

        X = data[cols].values


        model = make_pipeline(
            StandardScaler(),

            LogisticRegression(
                max_iter=2000,
                multi_class="multinomial",
                solver="lbfgs"
            )
        )


        accuracy = cross_val_score(
            model,
            X,
            y,
            cv=gkf,
            groups=groups,
            scoring="f1_macro"
        )


        print(
            f"{name}: "
            f"Accuracy = {accuracy.mean():.3f} ± {accuracy.std():.3f}"
        )
evaluate_hypertension()

Demographics only: Accuracy = 0.326 ± 0.100
Hand-crafted PPG feats: Accuracy = 0.294 ± 0.053
PaPaGei embeddings: Accuracy = 0.336 ± 0.045
All features: Accuracy = 0.350 ± 0.033


In [10]:
# Try much stronger regularization, tuned automatically per fold
alphas = [0.1, 1, 10, 100, 1000, 10000]
groups = data["subject_id"].values
# Just stronger Ridge alpha, no dimensionality reduction
X = data[papagei_cols + ["BMI", "Age"]].values
pipe = make_pipeline(StandardScaler(), RidgeCV(alphas=alphas))
r2 = cross_val_score(pipe, X, y, cv=gkf, groups=groups, scoring="r2")
print(f"PaPaGei + RidgeCV (auto-tuned alpha): R² = {r2.mean():.3f} ± {r2.std():.3f}")

# PCA down to far fewer components first, THEN Ridge
for n_comp in [5, 10, 20, 50, 75, 100, 150, 200]:
    pipe = make_pipeline(
        StandardScaler(),
        PCA(n_components=n_comp),
        RidgeCV(alphas=alphas)
    )
    r2 = cross_val_score(pipe, X, y, cv=gkf, groups=groups, scoring="r2")
    print(f"PaPaGei PCA({n_comp}) + RidgeCV: R² = {r2.mean():.3f} ± {r2.std():.3f}")

PaPaGei + RidgeCV (auto-tuned alpha): R² = 0.143 ± 0.156
PaPaGei PCA(5) + RidgeCV: R² = -0.005 ± 0.101
PaPaGei PCA(10) + RidgeCV: R² = -0.011 ± 0.061
PaPaGei PCA(20) + RidgeCV: R² = 0.116 ± 0.110
PaPaGei PCA(50) + RidgeCV: R² = 0.152 ± 0.167
PaPaGei PCA(75) + RidgeCV: R² = 0.139 ± 0.160
PaPaGei PCA(100) + RidgeCV: R² = 0.141 ± 0.159
PaPaGei PCA(150) + RidgeCV: R² = 0.143 ± 0.154
PaPaGei PCA(200) + RidgeCV: R² = 0.143 ± 0.156


In [11]:
data = features_df.dropna(subset=["DBP"]).copy()

y = data["DBP"].values
groups = data["subject_id"].values


papagei_features = papagei_cols

other_features = (
    hand_crafted_cols +
    ["BMI", "Age"]
)


X = data[papagei_features + other_features]


# PCA only on Papagei embeddings
preprocessor = ColumnTransformer(

    transformers=[

        (
            "papagei",
            Pipeline([
                ("scale", StandardScaler()),
                ("pca", PCA(n_components=50))
            ]),
            papagei_features
        ),


        (
            "other",
            StandardScaler(),
            other_features
        )
    ]
)
model = Pipeline([
    ("preprocess", preprocessor),
    ("ridge",
     RidgeCV(
        alphas=[0.1,1,10,100,1000,10000]
     ))
])

gkf = GroupKFold(n_splits=5)

r2_scores = cross_val_score(
    model,
    X,
    y,
    cv=gkf,
    groups=groups,
    scoring="r2"
)

print(
    f"PaPaGei PCA(50) + Handcrafted + Demographics: "
    f"R² = {r2_scores.mean():.3f} ± {r2_scores.std():.3f}"
)


PaPaGei PCA(50) + Handcrafted + Demographics: R² = 0.038 ± 0.100


## ML MODELS ##

In [12]:
models = {
    "LightGBM": LGBMRegressor(
        n_estimators=300,
        learning_rate=0.03,
        num_leaves=15,
        max_depth=4,
        min_child_samples=30,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.5,
        reg_lambda=1,
        random_state=42,
        verbosity=-1),


    "XGBoost": XGBRegressor(
        n_estimators=500,
        learning_rate=0.03,
        max_depth=5,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42
    ),


    "CatBoost": CatBoostRegressor(
        iterations=500,
        learning_rate=0.03,
        depth=6,
        loss_function="RMSE",
        verbose=0,
        random_seed=42
    )

}

In [13]:
# Remove missing DBP
data = features_df.dropna(subset=["DBP"]).copy()

X_papagei = data[papagei_cols]


scaler = StandardScaler()

X_scaled = scaler.fit_transform(
    X_papagei
)


pca = PCA(
    n_components=50,
    random_state=42
)

X_pca = pca.fit_transform(
    X_scaled
)


# Create PCA dataframe
pca_df = pd.DataFrame(
    X_pca,
    columns=[f"pca_{i}" for i in range(50)],
    index=data.index
)


# Add PCA features to original dataframe
data = pd.concat(
    [
        data,
        pca_df
    ],
    axis=1
)


# -----------------------------
# Final feature selection
# -----------------------------

feature_cols = (
    [f"pca_{i}" for i in range(50)]
    +
    hand_crafted_cols
    +
    ["BMI", "Age"]
)


X = data[feature_cols]

y = data["DBP"].values

groups = data["subject_id"].values


print("Feature shape:", X.shape)
print("Number of subjects:", len(set(groups)))

Feature shape: (657, 74)
Number of subjects: 219


In [14]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

gkf = GroupKFold(n_splits=5)

results = {}
prediction_results = {}

for name, model in models.items():

    print("\n====================")
    print(name)
    print("====================")

    r2_scores = []
    mae_scores = []
    rmse_scores = []

    # Store predictions for this model
    fold_predictions = []

    for fold, (train_idx, test_idx) in enumerate(
        gkf.split(X, y, groups)
    ):

        print(f"Fold: {fold+1}")

        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]

        y_train = y[train_idx]
        y_test = y[test_idx]

        # Train
        model.fit(X_train, y_train)

        # Predict
        pred = model.predict(X_test)

        # Metrics
        r2 = r2_score(y_test, pred)
        mae = mean_absolute_error(y_test, pred)
        rmse = np.sqrt(mean_squared_error(y_test, pred))

        r2_scores.append(r2)
        mae_scores.append(mae)
        rmse_scores.append(rmse)

        print(
            f"R²={r2:.3f}, "
            f"MAE={mae:.2f}, "
            f"RMSE={rmse:.2f}"
        )

        # Save predictions for this fold
        temp = features_df.iloc[test_idx][
            ["subject_id", "Age", "BMI", "Hypertension"]
        ].copy()

        temp["Fold"] = fold + 1
        temp["True_DBP"] = y_test
        temp["Pred_DBP"] = pred
        temp["Absolute_Error"] = np.abs(y_test - pred)

        fold_predictions.append(temp)

    # Combine all folds
    prediction_results[name] = pd.concat(
        fold_predictions,
        ignore_index=True
    )

    # Average metrics
    results[name] = {
        "R2": np.mean(r2_scores),
        "MAE": np.mean(mae_scores),
        "RMSE": np.mean(rmse_scores)
    }

print("\nFINAL RESULTS")
results_df = pd.DataFrame(results).T
print(results_df)


LightGBM
Fold: 1
R²=0.121, MAE=7.28, RMSE=9.95
Fold: 2
R²=-0.142, MAE=8.64, RMSE=10.12
Fold: 3
R²=-0.030, MAE=9.67, RMSE=11.97
Fold: 4
R²=-0.002, MAE=9.14, RMSE=11.73
Fold: 5
R²=-0.073, MAE=8.89, RMSE=11.44

XGBoost
Fold: 1
R²=0.142, MAE=7.29, RMSE=9.83
Fold: 2
R²=-0.122, MAE=8.53, RMSE=10.03
Fold: 3
R²=-0.002, MAE=9.68, RMSE=11.80
Fold: 4
R²=0.011, MAE=9.07, RMSE=11.65
Fold: 5
R²=-0.016, MAE=8.48, RMSE=11.13

CatBoost
Fold: 1
R²=0.181, MAE=7.33, RMSE=9.61
Fold: 2
R²=-0.083, MAE=8.37, RMSE=9.85
Fold: 3
R²=0.019, MAE=9.43, RMSE=11.68
Fold: 4
R²=0.085, MAE=8.57, RMSE=11.21
Fold: 5
R²=0.002, MAE=8.56, RMSE=11.03

FINAL RESULTS
                R2       MAE       RMSE
LightGBM -0.024981  8.725012  11.040669
XGBoost   0.002699  8.610856  10.889853
CatBoost  0.040689  8.452964  10.677034


In [15]:
# Remove missing SBP
data = features_df.dropna(subset=["SBP"]).copy()

X_papagei = data[papagei_cols]


scaler = StandardScaler()

X_scaled = scaler.fit_transform(
    X_papagei
)


pca = PCA(
    n_components=50,
    random_state=42
)

X_pca = pca.fit_transform(
    X_scaled
)


# Create PCA dataframe
pca_df = pd.DataFrame(
    X_pca,
    columns=[f"pca_{i}" for i in range(50)],
    index=data.index
)


# Add PCA features to original dataframe
data = pd.concat(
    [
        data,
        pca_df
    ],
    axis=1
)


# -----------------------------
# Final feature selection
# -----------------------------

feature_cols = (
    [f"pca_{i}" for i in range(50)]
    +
    hand_crafted_cols
    +
    ["BMI", "Age"]
)


X = data[feature_cols]

y = data["SBP"].values

groups = data["subject_id"].values


print("Feature shape:", X.shape)
print("Number of subjects:", len(set(groups)))

Feature shape: (657, 74)
Number of subjects: 219


In [16]:
gkf = GroupKFold(n_splits=5)


results = {}


for name, model in models.items():

    print("\n====================")
    print(name)
    print("====================")


    r2_scores = []
    mae_scores = []
    rmse_scores = []


    for fold, (train_idx, test_idx) in enumerate(
        gkf.split(X, y, groups)
    ):

        print("Fold:", fold+1)


        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]


        y_train = y[train_idx]
        y_test = y[test_idx]


        # train
        model.fit(
            X_train,
            y_train
        )


        # predict
        pred = model.predict(
            X_test
        )


        # metrics

        r2 = r2_score(
            y_test,
            pred
        )


        mae = mean_absolute_error(
            y_test,
            pred
        )


        rmse = np.sqrt(
            mean_squared_error(
                y_test,
                pred
            )
        )


        r2_scores.append(r2)
        mae_scores.append(mae)
        rmse_scores.append(rmse)


        print(
            f"R²={r2:.3f}, "
            f"MAE={mae:.2f}, "
            f"RMSE={rmse:.2f}"
        )



    results[name] = {

        "R2": np.mean(r2_scores),
        "MAE": np.mean(mae_scores),
        "RMSE": np.mean(rmse_scores)

    }


print("\nFINAL RESULTS")

pd.DataFrame(results).T


LightGBM
Fold: 1
R²=0.181, MAE=14.08, RMSE=18.02
Fold: 2
R²=-0.175, MAE=12.58, RMSE=15.93
Fold: 3
R²=0.208, MAE=15.24, RMSE=18.93
Fold: 4
R²=0.169, MAE=15.87, RMSE=19.37
Fold: 5
R²=0.133, MAE=17.28, RMSE=21.04

XGBoost
Fold: 1
R²=0.241, MAE=13.46, RMSE=17.36
Fold: 2
R²=-0.077, MAE=12.26, RMSE=15.25
Fold: 3
R²=0.222, MAE=14.78, RMSE=18.77
Fold: 4
R²=0.155, MAE=15.76, RMSE=19.53
Fold: 5
R²=0.136, MAE=17.37, RMSE=20.99

CatBoost
Fold: 1
R²=0.238, MAE=13.64, RMSE=17.39
Fold: 2
R²=-0.091, MAE=12.25, RMSE=15.35
Fold: 3
R²=0.196, MAE=14.85, RMSE=19.08
Fold: 4
R²=0.153, MAE=15.94, RMSE=19.54
Fold: 5
R²=0.147, MAE=17.20, RMSE=20.87

FINAL RESULTS


,R2,MAE,RMSE
LightGBM,0.103165,15.007604,18.656977
XGBoost,0.135388,14.726955,18.377920
CatBoost,0.128522,14.775661,18.444995


## Neural Network ##

In [17]:
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score, mean_absolute_error

In [18]:
class BPNet(nn.Module):

    def __init__(self,input_dim):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(input_dim,128),
            nn.ReLU(),

            nn.Linear(128,64),
            nn.ReLU(),

            nn.Linear(64,32),
            nn.ReLU(),

            nn.Linear(32,1)
        )


    def forward(self,x):
        return self.network(x)

In [19]:
class BPNet(nn.Module):

    def __init__(self, input_dim):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),


            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),


            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),


            nn.Linear(128, 64),
            nn.ReLU(),


            nn.Linear(64, 1)
        )


    def forward(self, x):
        return self.network(x)

In [20]:
def train_model(X_train, y_train, X_val, y_val):

    device = "cuda" if torch.cuda.is_available() else "cpu"

    print("Training on:", device)


    model = BPNet(
        input_dim=X_train.shape[1]
    ).to(device)



    criterion = nn.MSELoss()


    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=1e-4,
        weight_decay=1e-4
    )


    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=10
    )


    X_train = torch.tensor(
        X_train,
        dtype=torch.float32
    ).to(device)


    y_train = torch.tensor(
        y_train,
        dtype=torch.float32
    ).view(-1,1).to(device)



    X_val = torch.tensor(
        X_val,
        dtype=torch.float32
    ).to(device)


    y_val = torch.tensor(
        y_val,
        dtype=torch.float32
    ).view(-1,1).to(device)



    epochs = 100


    best_loss = np.inf
    patience = 25
    counter = 0


    for epoch in range(epochs):

        # -----------------
        # Training
        # -----------------

        model.train()

        optimizer.zero_grad()


        pred = model(X_train)


        train_loss = criterion(
            pred,
            y_train
        )


        train_loss.backward()

        optimizer.step()



        # -----------------
        # Validation
        # -----------------

        model.eval()

        with torch.no_grad():

            val_pred = model(X_val)

            val_loss = criterion(
                val_pred,
                y_val
            )


        scheduler.step(val_loss)



        # print every epoch
        print(
            f"Epoch [{epoch+1}/{epochs}] "
            f"Train Loss: {train_loss.item():.4f} "
            f"Val Loss: {val_loss.item():.4f}"
        )


        # Early stopping
        if val_loss < best_loss:

            best_loss = val_loss
            counter = 0

            best_model = model.state_dict()


        else:

            counter += 1


        if counter >= patience:

            print("Early stopping")
            break



    model.load_state_dict(best_model)


    model.eval()

    with torch.no_grad():

        predictions = model(X_val).cpu().numpy()


    return predictions

## DBP ##

In [21]:
data = features_df.dropna(subset=["DBP"]).copy()

feature_cols = (
    papagei_cols +
    hand_crafted_cols +
    ["BMI", "Age"]
)

X = data[feature_cols].values
y =features_df["DBP"].values


groups = data["subject_id"].values

In [22]:
gkf = GroupKFold(n_splits=5)


r2_scores = []
mae_scores = []


for fold, (train_idx, test_idx) in enumerate(
        gkf.split(X, y, groups)):


    print("Fold:", fold+1)


    X_train = X[train_idx]
    X_test = X[test_idx]

    y_train = y[train_idx]
    y_test = y[test_idx]


    # normalize only using training data

    scaler = StandardScaler()

    X_train = scaler.fit_transform(X_train)

    X_test = scaler.transform(X_test)

    predictions = train_model(
        X_train,
        y_train,
        X_test,
        y_test
    )


    r2 = r2_score(
        y_test,
        predictions
    )

    mae = mean_absolute_error(
        y_test,
        predictions
    )


    r2_scores.append(r2)
    mae_scores.append(mae)


    print(
        "R2:",
        round(r2,3),
        "MAE:",
        round(mae,3)
    )



print("\nFinal Results")

print(
    "R2:",
    np.mean(r2_scores),
    "+/-",
    np.std(r2_scores)
)

print(
    "MAE:",
    np.mean(mae_scores)
)

Fold: 1
Training on: cuda
Epoch [1/100] Train Loss: 5292.6084 Val Loss: 5191.9082
Epoch [2/100] Train Loss: 5288.8979 Val Loss: 5191.9165
Epoch [3/100] Train Loss: 5287.7393 Val Loss: 5191.5674
Epoch [4/100] Train Loss: 5284.5503 Val Loss: 5190.8716
Epoch [5/100] Train Loss: 5281.0811 Val Loss: 5189.8257
Epoch [6/100] Train Loss: 5280.6660 Val Loss: 5188.6240
Epoch [7/100] Train Loss: 5277.5674 Val Loss: 5187.1963
Epoch [8/100] Train Loss: 5273.8477 Val Loss: 5185.6401
Epoch [9/100] Train Loss: 5271.5220 Val Loss: 5184.0171
Epoch [10/100] Train Loss: 5268.9297 Val Loss: 5182.1279
Epoch [11/100] Train Loss: 5266.9683 Val Loss: 5180.0894
Epoch [12/100] Train Loss: 5263.4331 Val Loss: 5177.9028
Epoch [13/100] Train Loss: 5261.1274 Val Loss: 5175.5596
Epoch [14/100] Train Loss: 5257.8145 Val Loss: 5173.0562
Epoch [15/100] Train Loss: 5255.4199 Val Loss: 5170.3608
Epoch [16/100] Train Loss: 5253.6411 Val Loss: 5167.5981
Epoch [17/100] Train Loss: 5250.0884 Val Loss: 5164.7573
Epoch [18/100]

## SBP ##

In [23]:
data = features_df.dropna(subset=["SBP"]).copy()

feature_cols = (
    papagei_cols +
    hand_crafted_cols +
    ["BMI", "Age"]
)

X = data[feature_cols].values
y= data["SBP"].values


groups = data["subject_id"].values

In [24]:
gkf = GroupKFold(n_splits=5)


r2_scores = []
mae_scores = []


for fold, (train_idx, test_idx) in enumerate(
        gkf.split(X, y, groups)):


    print("Fold:", fold+1)


    X_train = X[train_idx]
    X_test = X[test_idx]

    y_train = y[train_idx]
    y_test = y[test_idx]


    # normalize only using training data

    scaler = StandardScaler()

    X_train = scaler.fit_transform(X_train)

    X_test = scaler.transform(X_test)

    predictions = train_model(
        X_train,
        y_train,
        X_test,
        y_test
    )


    r2 = r2_score(
        y_test,
        predictions
    )

    mae = mean_absolute_error(
        y_test,
        predictions
    )


    r2_scores.append(r2)
    mae_scores.append(mae)


    print(
        "R2:",
        round(r2,3),
        "MAE:",
        round(mae,3)
    )



print("\nFinal Results")

print(
    "R2:",
    np.mean(r2_scores),
    "+/-",
    np.std(r2_scores)
)

print(
    "MAE:",
    np.mean(mae_scores)
)

Fold: 1
Training on: cuda
Epoch [1/100] Train Loss: 16745.6797 Val Loss: 17181.4238
Epoch [2/100] Train Loss: 16740.4375 Val Loss: 17181.0039
Epoch [3/100] Train Loss: 16736.1426 Val Loss: 17180.8496
Epoch [4/100] Train Loss: 16731.5488 Val Loss: 17181.0762
Epoch [5/100] Train Loss: 16727.5918 Val Loss: 17181.3984
Epoch [6/100] Train Loss: 16721.4922 Val Loss: 17181.8613
Epoch [7/100] Train Loss: 16715.8359 Val Loss: 17182.0801
Epoch [8/100] Train Loss: 16712.8203 Val Loss: 17181.9805
Epoch [9/100] Train Loss: 16708.2695 Val Loss: 17181.5488
Epoch [10/100] Train Loss: 16707.5098 Val Loss: 17180.9238
Epoch [11/100] Train Loss: 16702.6504 Val Loss: 17179.9941
Epoch [12/100] Train Loss: 16697.6855 Val Loss: 17178.7734
Epoch [13/100] Train Loss: 16692.7090 Val Loss: 17177.1738
Epoch [14/100] Train Loss: 16691.2383 Val Loss: 17175.3828
Epoch [15/100] Train Loss: 16685.8164 Val Loss: 17173.1855
Epoch [16/100] Train Loss: 16681.8809 Val Loss: 17170.6699
Epoch [17/100] Train Loss: 16679.4707 V

## Comparing Handcrafted vs PaPaGei vs Combined Features ##

In [25]:
alphas = [0.1, 1, 10, 100, 1000]

feature_sets = {
    "Handcrafted": hand_crafted_cols + ["Age", "BMI"],
    "PaPaGei": papagei_cols + ["Age", "BMI"],
    "Combined": hand_crafted_cols + papagei_cols + ["Age", "BMI"]
}

gkf = GroupKFold(n_splits=5)

for name, cols in feature_sets.items():

    X = features_df[cols]
    y = features_df["DBP"].values
    groups = features_df["subject_id"].values

    r2_scores = []
    mae_scores = []
    rmse_scores = []

    for train_idx, test_idx in gkf.split(X, y, groups):

        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]

        y_train = y[train_idx]
        y_test = y[test_idx]

        if name == "Handcrafted":

            pipe = Pipeline([
                ("scaler", StandardScaler()),
                ("ridge", RidgeCV(alphas=alphas))
            ])

        else:

            pipe = Pipeline([
                ("scaler", StandardScaler()),
                ("pca", PCA(n_components=50)),
                ("ridge", RidgeCV(alphas=alphas))
            ])

        pipe.fit(X_train, y_train)

        pred = pipe.predict(X_test)

        r2_scores.append(r2_score(y_test, pred))
        mae_scores.append(mean_absolute_error(y_test, pred))
        rmse_scores.append(np.sqrt(mean_squared_error(y_test, pred)))

    print("\n", name)
    print("R²   :", np.mean(r2_scores))
    print("MAE  :", np.mean(mae_scores))
    print("RMSE :", np.mean(rmse_scores))


 Handcrafted
R²   : 0.0315905105724902
MAE  : 8.520772611253117
RMSE : 10.74067811413984

 PaPaGei
R²   : 0.034946390228250655
MAE  : 8.519988661355814
RMSE : 10.710081201971592

 Combined
R²   : 0.03810827912896253
MAE  : 8.547741202298765
RMSE : 10.698229307741984


## Error vs Age ##

To check if the model performs better for a certain age group

In [26]:
age_bins = [0, 30, 40, 50, 60, 100]
age_labels = ["<30", "30-40", "40-50", "50-60", "60+"]

for model_name in ["LightGBM", "XGBoost", "CatBoost"]:

    df = prediction_results[model_name].copy()

    df["Age_Group"] = pd.cut(
        df["Age"],
        bins=age_bins,
        labels=age_labels
    )

    print("\n==============================")
    print(model_name)
    print("Error by Age Group")
    print("==============================")

    print(
        df.groupby("Age_Group")["Absolute_Error"]
          .agg(["mean", "std", "count"])
          .round(2)
    )


LightGBM
Error by Age Group
           mean   std  count
Age_Group                   
<30        7.43  4.49     75
30-40      8.43  5.68     15
40-50      9.23  8.08     87
50-60      9.55  6.98    186
60+        8.40  6.82    294

XGBoost
Error by Age Group
           mean   std  count
Age_Group                   
<30        6.70  4.85     75
30-40      9.22  6.33     15
40-50      9.17  7.82     87
50-60      9.85  6.87    186
60+        8.12  6.57    294

CatBoost
Error by Age Group
           mean   std  count
Age_Group                   
<30        6.58  4.55     75
30-40      9.18  7.29     15
40-50      9.02  7.53     87
50-60      9.62  6.70    186
60+        7.99  6.47    294


## Error vs BMI ##

To check if the model performs better for a certain for people lying in specific BMI groups

In [27]:
bmi_bins = [0, 18.5, 25, 30, 100]
bmi_labels = ["Underweight", "Normal", "Overweight", "Obese"]

for model_name in ["LightGBM", "XGBoost", "CatBoost"]:

    df = prediction_results[model_name].copy()

    df["BMI_Group"] = pd.cut(
        df["BMI"],
        bins=bmi_bins,
        labels=bmi_labels
    )

    print("\n==============================")
    print(model_name)
    print("Error by BMI Group")
    print("==============================")

    print(
        df.groupby("BMI_Group")["Absolute_Error"]
          .agg(["mean", "std", "count"])
          .round(2)
    )


LightGBM
Error by BMI Group
             mean   std  count
BMI_Group                     
Underweight  7.47  5.03     57
Normal       9.14  7.31    438
Overweight   8.09  5.33    126
Obese        7.85  7.47     36

XGBoost
Error by BMI Group
             mean   std  count
BMI_Group                     
Underweight  6.87  5.53     57
Normal       9.03  7.14    438
Overweight   8.20  5.40    126
Obese        7.73  6.91     36

CatBoost
Error by BMI Group
             mean   std  count
BMI_Group                     
Underweight  7.15  5.67     57
Normal       8.82  7.01    438
Overweight   7.93  5.16    126
Obese        7.89  6.72     36


In [28]:
# # Creating one representative sample per subject by aaveraging all segments for that subject
# hand_crafted_cols = ["mean_ppg","std_ppg","skew_ppg","kurtosis_ppg","rms_ppg","max_ppg",
#     "min_ppg","range_ppg","heart_rate","mean_peak_amp","std_peak_amp","mean_peak_distance",
#     "max_vpg","min_vpg","mean_vpg","std_vpg","max_apg","min_apg","mean_apg","std_apg",
#     "dominant_freq","spectral_energy"]
# papagei_cols = [c for c in features_df.columns if c.startswith("papagei_")]

# subject_df = features_df.groupby("subject_id")[hand_crafted_cols + papagei_cols].mean().reset_index()
# labels = features_df.drop_duplicates("subject_id")[["subject_id","SBP","DBP","Age","BMI","Hypertension"]]
# subject_df = subject_df.merge(labels, on="subject_id")
# print(subject_df.shape)